In [ ]:
!pip install ultralytics roboflow opencv-python-headless

Master Code (just run cell 1 and 2)

In [ ]:
import os
import glob
import cv2
import numpy as np
import time
import math
import zipfile
from collections import defaultdict, deque
from ultralytics import YOLO

print("\n=====================================================================")
print("🚀 ADAPTIVE MULTI-SCENARIO)
print("=====================================================================")

# ==========================================
# 1. SETUP
# ==========================================
WORKING_DIR = "/kaggle/working"
VIOLATION_DIR = os.path.join(WORKING_DIR, "violations")
os.makedirs(VIOLATION_DIR, exist_ok=True)
print(f"📁 Folder penyimpanan capture pelanggaran: {VIOLATION_DIR}")

# --- PATH DIRECTORY ---
INPUT_VIDEO_PATH = "/kaggle/input/datasets/darnisaazzahra/etle-test-video/Test_video_1_20250805_125220.mp4"

# Cek apakah file benar-benar ada di path tersebut
if not os.path.exists(INPUT_VIDEO_PATH):
    print(f"⚠️ Video tidak ditemukan di: {INPUT_VIDEO_PATH}")
    exit()

video_filename = os.path.basename(INPUT_VIDEO_PATH)
ZIP_OUTPUT_PATH = os.path.join(WORKING_DIR, f"violations_{video_filename.split('.')[0]}.zip")
print(f"🎥 Memproses video: {INPUT_VIDEO_PATH}")

# Load Models
print("\n⏳ Loading Models...")
model_standard = YOLO("yolov8m.pt") 
trained_model_path = "/kaggle/input/datasets/darnisaazzahra/modeletleyolo/best.pt" 
model_custom = YOLO(trained_model_path)


# ==========================================
# 2. GLOBAL FUNCTIONS
# ==========================================
def get_traffic_light_state(crop_img):
    """Fungsi deteksi warna lampu (digunakan di Skenario 1 & 2)"""
    if crop_img.size == 0: return "UNKNOWN"
    blurred = cv2.GaussianBlur(crop_img, (5, 5), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    
    mask_red1 = cv2.inRange(hsv, np.array([0, 40, 120]), np.array([10, 255, 255]))
    mask_red2 = cv2.inRange(hsv, np.array([160, 40, 120]), np.array([180, 255, 255]))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)
    mask_green = cv2.inRange(hsv, np.array([40, 40, 120]), np.array([90, 255, 255]))
    
    r_cnt, g_cnt = cv2.countNonZero(mask_red), cv2.countNonZero(mask_green)
    if r_cnt > g_cnt and r_cnt > 10: return "RED"
    elif g_cnt > r_cnt and g_cnt > 10: return "GREEN"
    return "UNKNOWN"

def create_violation_zip(source_dir, output_zip):
    """Fungsi kompresi folder hasil capture ke ZIP"""
    print(f"\n📦 Mengemas seluruh capture pelanggaran ke dalam {output_zip}...")
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                if file.endswith(('.jpg', '.png')):
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, start=source_dir)
                    zipf.write(file_path, arcname)
    print("✅ Kompresi ZIP selesai!")


# ==========================================
# 3. QUICK SCAN (ANALISIS KONDISI AWAL)
# ==========================================
def quick_scan_video(video_path):
    print("\n🔍 Memulai Quick Scan (Menganalisis 30 frame pertama)...")
    cap = cv2.VideoCapture(video_path)
    
    has_tl = False
    max_tl_count = 0  # <-- Variabel baru untuk menghitung jumlah lampu
    has_zebra = False
    max_zebra_count = 0 # <-- Variabel baru untuk menghitung jumlah zebra cross
    is_dynamic = False
    
    ret, old_frame = cap.read()
    if not ret: return 3
    
    # Setup parameter Optical Flow untuk deteksi pergerakan kamera
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)
    
    frame_count = 0
    motion_accum = 0.0
    
    while cap.isOpened() and frame_count < 30:
        ret, frame = cap.read()
        if not ret: break
        
        # Cek Keberadaan Traffic Light (Class 9 di YOLO Standard)
        res_std = model_standard.predict(frame, verbose=False)
        if res_std[0].boxes is not None:
            classes = res_std[0].boxes.cls.cpu().numpy().astype(int)
            tl_count_in_frame = np.count_nonzero(classes == 9)
            if tl_count_in_frame > 0:
                has_tl = True
                if tl_count_in_frame > max_tl_count:
                    max_tl_count = tl_count_in_frame # Update jumlah maksimal TL yg terdeteksi
                
        # Cek Keberadaan Zebra Cross (YOLO Custom)
        res_cstm = model_custom.predict(frame, conf=0.30, verbose=False)
        if res_cstm[0].boxes is not None and len(res_cstm[0].boxes) > 0:
            has_zebra = True
            zebra_count_in_frame = len(res_cstm[0].boxes)
            if zebra_count_in_frame > max_zebra_count:
                max_zebra_count = zebra_count_in_frame # Update jumlah maksimal Zebra yg terdeteksi
            
        # Kalkulasi pergerakan kamera via Optical Flow
        if p0 is not None:
            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
            if p1 is not None and st is not None:
                good_new, good_old = p1[st == 1], p0[st == 1]
                if len(good_new) > 0:
                    distances = np.linalg.norm(good_new - good_old, axis=1)
                    motion_accum += float(np.median(distances))
                p0 = good_new.reshape(-1, 1, 2)
            old_gray = frame_gray.copy()
            
        frame_count += 1
        
    cap.release()
    
    avg_motion = motion_accum / frame_count if frame_count > 0 else 0
    if avg_motion > 1.5: 
        is_dynamic = True
        
    print(f"   ↳ Traffic Light: {has_tl} (Jumlah: {max_tl_count}) | Zebra Cross: {has_zebra} (Jumlah: {max_zebra_count}) | Kamera Dinamis: {is_dynamic} (Motion Score: {avg_motion:.2f})")
    
     # --- LOGIK TAMBAHAN UNTUK SKENARIO 3 ---
    if has_tl and max_tl_count == 1:
        print("   👉 KEPUTUSAN: Menjalankan SKENARIO 3 (Hanya ada 1 Lampu Lalu Lintas yang terdeteksi)")
        return 3
    # ---------------------------------------
    elif is_dynamic or (has_tl and not has_zebra):
        print("   👉 KEPUTUSAN: Menjalankan SKENARIO 2 (Isolasi Lajur Ekstra/Kamera Dinamis)")
        return 2
    elif has_tl and has_zebra:
        # --- LOGIKA BARU UNTUK ZEBRA CROSS ---
        if max_zebra_count > 2:
            print(f"   👉 KEPUTUSAN: Menjalankan SKENARIO 1 (Terdapat {max_zebra_count} Zebra Cross - Lebih dari 2)")
            return 1
        elif max_zebra_count < 3:
            print("   👉 KEPUTUSAN: Menjalankan SKENARIO 3 (Hanya ada 1 atau 2 Zebra Cross yang terdeteksi)")
            return 3
        else:
            print("   👉 KEPUTUSAN: Menjalankan SKENARIO 1 (Trajectory Directional Standar)")
            return 1
    else:
        print("   👉 KEPUTUSAN: Menjalankan SKENARIO 3 (Pure Behavioral Consensus)")
        return 3

# Jalankan Quick Scan
scenario_mode = quick_scan_video(INPUT_VIDEO_PATH)


# ==========================================
# 4. EKSEKUSI SKENARIO
# ==========================================

if scenario_mode == 1:
    print("\n=== MENJALANKAN SKENARIO 1 ===")
    OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, f"scenario1_{video_filename}")
    
    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    vehicle_trajectory = defaultdict(lambda: deque(maxlen=15)) 
    vehicle_zone_status = defaultdict(lambda: False)
    violation_ids = set()
    zone_light_history = defaultdict(lambda: deque(maxlen=20))
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1

        results_cw = model_custom.predict(frame, conf=0.35, verbose=False)
        zones = []
        
        for r in results_cw:
            for box in r.boxes:
                zx1, zy1, zx2, zy2 = map(int, box.xyxy[0])
                zc_x, zc_y = int((zx1 + zx2) / 2), int((zy1 + zy2) / 2)
                zones.append({'box': (zx1, zy1, zx2, zy2), 'center': (zc_x, zc_y), 'light_state': "UNKNOWN", 'best_light_pos': None})
                cv2.rectangle(frame, (zx1, zy1), (zx2, zy2), (255, 255, 0), 2)
                cv2.putText(frame, "ZONE", (zx1, zy1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        results_std = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
        
        if results_std[0].boxes is not None:
            boxes = results_std[0].boxes.xyxy.cpu().numpy()
            class_ids = results_std[0].boxes.cls.cpu().numpy().astype(int)
            
            lights = []
            for box, cls_id in zip(boxes, class_ids):
                if cls_id == 9: 
                    lx1, ly1, lx2, ly2 = map(int, box)
                    lc_x, lc_y = int((lx1 + lx2)/2), int((ly1 + ly2)/2)
                    state = get_traffic_light_state(frame[ly1:ly2, lx1:lx2])
                    lights.append({'center': (lc_x, lc_y), 'state': state})
                    cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), (0, 165, 255), 2)

            for idx, zone in enumerate(zones):
                zx, zy = zone['center']
                best_score = float('inf')
                best_light = None
                
                for light in lights:
                    lx, ly = light['center']
                    if ly < zy: 
                        dx, dy = abs(lx - zx), abs(ly - zy)
                        score = dx + (dy * 0.2) 
                        if score < best_score:
                            best_score = score
                            best_light = light
                
                if best_light:
                    zone_light_history[idx].append(best_light['state'])
                    zone['light_state'] = max(set(zone_light_history[idx]), key=zone_light_history[idx].count)
                    zone['best_light_pos'] = best_light['center']
                    cv2.line(frame, (zx, zy), best_light['center'], (255, 255, 255), 1)
                    l_color = (0,0,255) if zone['light_state']=="RED" else (0,255,0) if zone['light_state']=="GREEN" else (0,255,255)
                    cv2.putText(frame, f"TL:{zone['light_state']}", (zone['box'][0], zone['box'][1]+20), 0, 0.6, l_color, 2)

            if results_std[0].boxes.id is not None:
                track_ids = results_std[0].boxes.id.cpu().numpy().astype(int)
                for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
                    if cls_id in [2, 3, 5, 7]:
                        vx1, vy1, vx2, vy2 = map(int, box)
                        v_center_x, v_bottom_y = int((vx1+vx2)/2), vy2
                        
                        vehicle_trajectory[track_id].append((v_center_x, v_bottom_y))
                        moving_forward = False
                        if len(vehicle_trajectory[track_id]) > 5:
                            start_y = vehicle_trajectory[track_id][0][1]
                            current_y = vehicle_trajectory[track_id][-1][1]
                            if (start_y - current_y) > 5: 
                                moving_forward = True

                        for zone in zones:
                            zx1, zy1, zx2, zy2 = zone['box']
                            is_inside = (zx1 <= v_center_x <= zx2) and (zy1 <= v_bottom_y <= zy2)
                            was_inside = vehicle_zone_status[track_id]
                            
                            if zone['light_state'] == "RED" and is_inside and not was_inside and moving_forward:
                                if track_id not in violation_ids:
                                    violation_ids.add(track_id)
                                    
                                    # --- PASTIKAN BOUNDING BOX MERAH JELAS SAAT CAPTURE ---
                                    cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), (0, 0, 255), 3)
                                    cv2.putText(frame, "PELANGGARAN!", (vx1, vy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                                    # ------------------------------------------------------
                                    
                                    capture_name = os.path.join(VIOLATION_DIR, f"s1_violation_id{track_id}_frame{frame_idx}.jpg")
                                    cv2.imwrite(capture_name, frame)
                            
                            if is_inside:
                                vehicle_zone_status[track_id] = True
                        
                        is_currently_in_any = any((z['box'][0] <= v_center_x <= z['box'][2]) and (z['box'][1] <= v_bottom_y <= z['box'][3]) for z in zones)
                        if not is_currently_in_any:
                            vehicle_zone_status[track_id] = False

                        is_violator = track_id in violation_ids
                        color = (0, 0, 255) if is_violator else (0, 255, 0)
                        cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), color, 2)
                        
                        if moving_forward:
                            cv2.arrowedLine(frame, (v_center_x, v_bottom_y), (v_center_x, v_bottom_y - 20), (255, 255, 255), 2, tipLength=0.5)

        out.write(frame)

    cap.release()
    out.release()


elif scenario_mode == 2:
    print("\n=== MENJALANKAN SKENARIO 2 ===")
    OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, f"scenario2_{video_filename}")
    
    start_time = time.time()
    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps if fps > 0 else 30, (w, h))

    vehicle_trajectory = defaultdict(lambda: deque(maxlen=15)) 
    violation_ids = set()
    light_history = defaultdict(lambda: deque(maxlen=20))
    inferred_state_history = defaultdict(lambda: deque(maxlen=30)) 
    
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        results_cw = model_custom.predict(frame, conf=0.30, verbose=False)
        detected_zones = []
        if results_cw[0].boxes is not None:
            for box in results_cw[0].boxes:
                zx1, zy1, zx2, zy2 = map(int, box.xyxy[0])
                detected_zones.append({'box': (zx1, zy1, zx2, zy2)})

        results_std = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
        lights = []
        vehicles = [] 
        
        if results_std[0].boxes is not None:
            boxes_std = results_std[0].boxes.xyxy.cpu().numpy()
            classes_std = results_std[0].boxes.cls.cpu().numpy().astype(int)
            t_ids = results_std[0].boxes.id.cpu().numpy().astype(int) if results_std[0].boxes.id is not None else []
            
            for box, cls, t_id in zip(boxes_std, classes_std, t_ids if len(t_ids) > 0 else [-1]*len(boxes_std)):
                if cls == 9: 
                    lx1, ly1, lx2, ly2 = map(int, box)
                    state = get_traffic_light_state(frame[ly1:ly2, lx1:lx2])
                    lights.append({'box': (lx1, ly1, lx2, ly2), 'center': ((lx1+lx2)//2, ly2), 'state': state})
                elif cls in [2, 3, 5, 7] and t_id != -1: 
                    vx1, vy1, vx2, vy2 = map(int, box)
                    v_cx, v_by = (vx1+vx2)//2, vy2
                    vehicle_trajectory[t_id].append((v_cx, v_by))
                    
                    is_moving_forward = False
                    if len(vehicle_trajectory[t_id]) >= 3:
                        prev_y = vehicle_trajectory[t_id][-3][1]
                        is_moving_forward = (prev_y - v_by) > 2 
                        
                    vehicles.append({'id': t_id, 'box': (vx1, vy1, vx2, vy2), 'center_bottom': (v_cx, v_by), 'moving': is_moving_forward})

        active_violation_zones = []
        claimed_zone_indices = set()

        # Tahap 1
        for light in lights:
            lc_x, lc_y = light['center']
            s_id = f"L_{lc_x//50}_{lc_y//50}"
            light_history[s_id].append(light['state'])
            stable_state = max(set(light_history[s_id]), key=light_history[s_id].count)

            matched_zone_idx = -1
            v_box = None
            
            for idx, zone in enumerate(detected_zones):
                zx1, zy1, zx2, zy2 = zone['box']
                zc_x = (zx1 + zx2) // 2
                if abs(zc_x - lc_x) < (w * 0.20):
                    if lc_y < zy1 and (zy1 - lc_y) < (h * 0.45):
                        v_box = zone['box']
                        matched_zone_idx = idx
                        break
            
            if v_box is not None:
                claimed_zone_indices.add(matched_zone_idx)
                cv2.rectangle(frame, (v_box[0], v_box[1]), (v_box[2], v_box[3]), (255, 255, 0), 2)
                cv2.putText(frame, "REAL ZONE", (v_box[0], v_box[1] - 5), 0, 0.5, (255, 255, 0), 2)
            else:
                v_width = int(w * 0.25)
                v_y1 = lc_y + int(h * 0.25)
                v_y2 = v_y1 + int(h * 0.08)
                v_y1, v_y2 = min(v_y1, h - 5), min(v_y2, h)
                v_x1 = lc_x - (v_width // 2)
                v_x2 = lc_x + (v_width // 2)
                v_box = (v_x1, v_y1, v_x2, v_y2)
                cv2.rectangle(frame, (v_x1, v_y1), (v_x2, v_y2), (255, 100, 0), 2)
                cv2.putText(frame, "VIRTUAL ZONE", (v_x1, v_y1 - 5), 0, 0.4, (255, 100, 0), 2)

            active_violation_zones.append({'box': v_box, 'state': stable_state, 'source': 'PHYSICAL'})
            l_col = (0,0,255) if stable_state == "RED" else (0,255,0)
            cv2.rectangle(frame, (light['box'][0], light['box'][1]), (light['box'][2], light['box'][3]), l_col, 2)
            cv2.putText(frame, stable_state, (light['box'][0], light['box'][1]-10), 0, 0.6, l_col, 2)

        # Tahap 2
        for idx, zone in enumerate(detected_zones):
            if idx not in claimed_zone_indices:
                zx1, zy1, zx2, zy2 = zone['box']
                queue_area_y_end = min(zy1 + int(h * 0.25), h)
                
                stop_count, move_count = 0, 0
                for v in vehicles:
                    v_cx, v_by = v['center_bottom']
                    if zx1 <= v_cx <= zx2 and zy1 < v_by < queue_area_y_end:
                        if v['moving']: move_count += 1
                        else: stop_count += 1
                
                current_inferred_state = "GREEN"
                if stop_count > move_count and stop_count >= 1: 
                    current_inferred_state = "RED"
                
                z_id = f"Z_INF_{zx1}_{zy1}"
                inferred_state_history[z_id].append(current_inferred_state)
                stable_inferred = max(set(inferred_state_history[z_id]), key=inferred_state_history[z_id].count)

                active_violation_zones.append({'box': zone['box'], 'state': stable_inferred, 'source': 'INFERRED'})
                z_col = (0,0,255) if stable_inferred == "RED" else (0,255,0)
                cv2.rectangle(frame, (zx1, zy1), (zx2, zy2), z_col, 2)
                cv2.putText(frame, f"INFERRED: {stable_inferred}", (zx1, zy1 - 5), 0, 0.5, z_col, 2)

        for v in vehicles:
            t_id = v['id']
            vx1, vy1, vx2, vy2 = v['box']
            v_cx, v_by = v['center_bottom']
            is_moving_forward = v['moving']
            
            if len(vehicle_trajectory[t_id]) >= 3:
                prev_y = vehicle_trajectory[t_id][-3][1]
                for zone_info in active_violation_zones:
                    zx1, zy1, zx2, zy2 = zone_info['box']
                    
                    if zx1 <= v_cx <= zx2: 
                        if zone_info['state'] == "RED":
                            crossed_line = (prev_y > zy1 and v_by <= zy1)
                            moving_inside_zone = (zy1 <= v_by <= zy2 and is_moving_forward)
                            moving_past_zone = (v_by < zy1 and prev_y <= zy1 and is_moving_forward) 
                            
                            if crossed_line or moving_inside_zone or moving_past_zone:
                                if t_id not in violation_ids:
                                    violation_ids.add(t_id)
                                    
                                    # --- PASTIKAN BOUNDING BOX MERAH JELAS SAAT CAPTURE ---
                                    cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), (0, 0, 255), 3)
                                    cv2.putText(frame, "PELANGGARAN!", (vx1, vy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                                    # ------------------------------------------------------
                                    
                                    capture_name = os.path.join(VIOLATION_DIR, f"s2_violation_id{t_id}_frame{frame_count}.jpg")
                                    cv2.imwrite(capture_name, frame)

            color = (0, 0, 255) if t_id in violation_ids else (0, 255, 0)
            cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), color, 2)
            if t_id in violation_ids:
                cv2.putText(frame, "PELANGGARAN!", (vx1, vy1-10), 0, 0.6, (0,0,255), 2)

        cv2.putText(frame, f"Pelanggaran: {len(violation_ids)}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        out.write(frame)

    cap.release()
    out.release()


elif scenario_mode == 3:
    print("\n=== MENJALANKAN SKENARIO 3 ===")
    OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, f"scenario3_{video_filename}")

    vehicle_registry = defaultdict(lambda: deque(maxlen=30))
    smoothed_positions = {}
    stopped_frame_counter = defaultdict(int)
    confirmed_violations = set()
    persistent_anchors = {} 
    
    ANCHOR_TTL_FRAMES = 60 
    PROXIMITY_THRESHOLD = 0.35 
    STOP_SPEED_THRESHOLD = 0.5 
    STOP_DURATION_FRAMES = 10   
    MOVING_SPEED_THRESHOLD = 1.5 
    LINE_EXTENT = 250            
    SEGMENT_WIDTH_THRESHOLD = 0.15 

    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps if fps > 0 else 30, (w, h))

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1
        
        results = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
        active_vehicles = []
        
        if results[0].boxes is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy().astype(int)
            track_ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else []
            
            for idx, (box, cls) in enumerate(zip(boxes, classes)):
                if cls in [2, 3, 5, 7] and idx < len(track_ids): 
                    t_id = track_ids[idx]
                    x1, y1, x2, y2 = map(int, box)
                    raw_cx, raw_cy = (x1 + x2) // 2, y2 
                    
                    if t_id not in smoothed_positions: 
                        smoothed_positions[t_id] = (raw_cx, raw_cy)
                    else:
                        scx, scy = smoothed_positions[t_id]
                        smoothed_positions[t_id] = (int(scx * 0.7 + raw_cx * 0.3), int(scy * 0.7 + raw_cy * 0.3))
                    
                    cx, cy = smoothed_positions[t_id]
                    v_hist = vehicle_registry[t_id]
                    
                    speed, dominant_direction = 0.0, "UNKNOWN"
                    if len(v_hist) > 0:
                        past_cx, past_cy = v_hist[0]['cx'], v_hist[0]['cy']
                        dy, dx = cy - past_cy, cx - past_cx
                        speed = math.sqrt(dx**2 + dy**2) / len(v_hist)
                        
                        if abs(dy) > abs(dx):
                            dominant_direction = "UP_STREAM" if dy < 0 else "DOWN_STREAM"
                        else:
                            dominant_direction = "LEFT_STREAM" if dx < 0 else "RIGHT_STREAM"
                    
                    if speed < STOP_SPEED_THRESHOLD:
                        stopped_frame_counter[t_id] += 1
                    else:
                        stopped_frame_counter[t_id] = max(0, stopped_frame_counter[t_id] - 2)
                        
                    is_fully_stopped = (stopped_frame_counter[t_id] >= STOP_DURATION_FRAMES) and (speed < STOP_SPEED_THRESHOLD)
                    v_hist.append({'cx': cx, 'cy': cy, 'speed': speed, 'dir': dominant_direction})
                    
                    active_vehicles.append({
                        'id': t_id, 'box': (x1, y1, x2, y2), 'center': (cx, cy),
                        'speed': speed, 'direction': dominant_direction, 'is_stopped': is_fully_stopped
                    })

        processed_ids = set()
        dynamic_anchors = [] 

        for v in active_vehicles:
            if v['id'] in processed_ids: continue
            
            neighborhood = []
            for peer in active_vehicles:
                if peer['direction'] == v['direction'] and v['direction'] != "UNKNOWN":
                    dx = v['center'][0] - peer['center'][0]
                    dy = v['center'][1] - peer['center'][1]
                    dist = math.sqrt(dx**2 + dy**2)
                    
                    if dist < (w * PROXIMITY_THRESHOLD):
                        is_in_same_segment = True
                        if v['direction'] in ["UP_STREAM", "DOWN_STREAM"]:
                            if abs(dx) > (w * SEGMENT_WIDTH_THRESHOLD):
                                is_in_same_segment = False
                        else:
                            if abs(dy) > (h * SEGMENT_WIDTH_THRESHOLD):
                                is_in_same_segment = False
                        
                        if is_in_same_segment:
                            neighborhood.append(peer)
                            processed_ids.add(peer['id'])
                            
            stopped_cars = [c for c in neighborhood if c['is_stopped']]
            moving_cars = [c for c in neighborhood if not c['is_stopped']]
            
            if len(stopped_cars) >= 2 and len(stopped_cars) >= len(moving_cars):
                direction = v['direction']
                leader_car = None
                
                if direction == "UP_STREAM": leader_car = min(stopped_cars, key=lambda c: c['center'][1])
                elif direction == "DOWN_STREAM": leader_car = max(stopped_cars, key=lambda c: c['center'][1])
                elif direction == "LEFT_STREAM": leader_car = min(stopped_cars, key=lambda c: c['center'][0])
                elif direction == "RIGHT_STREAM": leader_car = max(stopped_cars, key=lambda c: c['center'][0])
                    
                if leader_car:
                    anchor_pos = leader_car['center']
                    dynamic_anchors.append({'direction': direction, 'pos': anchor_pos, 'neighborhood': neighborhood})
                    persistent_anchors[direction] = {'pos': anchor_pos, 'ttl': ANCHOR_TTL_FRAMES}

        active_directions_this_frame = [a['direction'] for a in dynamic_anchors]
        
        for direction, p_data in list(persistent_anchors.items()):
            if direction not in active_directions_this_frame:
                p_data['ttl'] -= 1
                if p_data['ttl'] <= 0:
                    del persistent_anchors[direction] 
                    continue
            
            if direction not in active_directions_this_frame:
                ax, ay = p_data['pos']
                cv2.line(frame, (ax - LINE_EXTENT, ay), (ax + LINE_EXTENT, ay), (0, 64, 255), 2) 
                cv2.putText(frame, f"STOP LINE MEMORY ({direction})", (ax - LINE_EXTENT + 10, ay - 10), 0, 0.5, (0, 64, 255), 1)

        for v in active_vehicles:
            t_id = v['id']
            x1, y1, x2, y2 = v['box']
            cx, cy = v['center']
            
            if t_id in confirmed_violations:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(frame, f"VIOLATOR #{t_id}", (x1, y1 - 10), 0, 0.5, (0, 0, 255), 2)
                continue
                
            if v['direction'] in persistent_anchors:
                p_anchor = persistent_anchors[v['direction']]
                ax, ay = p_anchor['pos']
                is_in_same_segment = False
                
                if v['direction'] in ["UP_STREAM", "DOWN_STREAM"]:
                    if abs(cx - ax) <= (w * SEGMENT_WIDTH_THRESHOLD): is_in_same_segment = True
                else:
                    if abs(cy - ay) <= (h * SEGMENT_WIDTH_THRESHOLD): is_in_same_segment = True
                
                if is_in_same_segment:
                    if not v['is_stopped'] and v['speed'] > MOVING_SPEED_THRESHOLD:
                        if stopped_frame_counter[t_id] > 0:
                            continue
                            
                        past_cy = vehicle_registry[t_id][0]['cy'] if len(vehicle_registry[t_id]) >= 5 else cy
                        past_cx = vehicle_registry[t_id][0]['cx'] if len(vehicle_registry[t_id]) >= 5 else cx
                        is_crossed = False
                        
                        if v['direction'] == "UP_STREAM" and past_cy >= ay and cy < ay: is_crossed = True
                        elif v['direction'] == "DOWN_STREAM" and past_cy <= ay and cy > ay: is_crossed = True
                        elif v['direction'] == "LEFT_STREAM" and past_cx >= ax and cx < ax: is_crossed = True
                        elif v['direction'] == "RIGHT_STREAM" and past_cx <= ax and cx > ax: is_crossed = True
                        
                        if is_crossed:
                            if t_id not in confirmed_violations:
                                confirmed_violations.add(t_id)
                                # --- PASTIKAN BOUNDING BOX MERAH JELAS SAAT CAPTURE ---
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                                cv2.putText(frame, f"PELANGGARAN #{t_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                                # ------------------------------------------------------
                                capture_filename = os.path.join(VIOLATION_DIR, f"s3_violation_id{t_id}_frame{frame_idx}.jpg")
                                cv2.imwrite(capture_filename, frame)

            if t_id not in confirmed_violations:
                color = (0, 255, 0) if not v['is_stopped'] else (255, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        cv2.rectangle(frame, (15, 15), (510, 85), (0, 0, 0), -1)
        cv2.rectangle(frame, (15, 15), (510, 85), (255, 128, 0), 2)
        cv2.putText(frame, "ETLE V8.0: MULTI-SCENARIO ADAPTIVE", (25, 40), 0, 0.5, (255, 128, 0), 2)
        cv2.putText(frame, f"ACTIVE ANCHORS: {len(persistent_anchors)} | VIOLATIONS: {len(confirmed_violations)}", (25, 70), 0, 0.5, (0, 0, 255), 2)

        out.write(frame)

    cap.release()
    out.release()

# ==========================================
# 5. FINALISASI (ZIP & REPORT)
# ==========================================
create_violation_zip(VIOLATION_DIR, ZIP_OUTPUT_PATH)
print(f"\n🎉 MASTER PROSES SELESAI!")
print(f"🎥 Video tersimpan di: {OUTPUT_VIDEO_PATH}")
print(f"📸 ZIP berisi semua capture pelanggaran ada di: {ZIP_OUTPUT_PATH}")

CODE per SKENARIO

In [ ]:
# WORKING_DIR = "/kaggle/working"
# trained_model_path = "/kaggle/input/datasets/darnisaazzahra/modeletleyolo/best.pt" 

In [ ]:
import os

#  menyimpan output
WORKING_DIR = "/kaggle/working"

INPUT_VIDEO_PATH = "/kaggle/input/etle-test-video/Test_video_1_20250805_125220.mp4" 

OUTPUT_MODEL_DIR = os.path.join(WORKING_DIR, "train_output")
OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, "etle_violation_output.mp4")

if not os.path.exists(OUTPUT_MODEL_DIR):
    os.makedirs(OUTPUT_MODEL_DIR)

# ============================================================
# INSTALASI & PERSIAPAN DATA (ROBOFLOW)
# ============================================================
print("\n=== STEP 1: MEMPERSIAPKAN DATA & LIBRARY ===")
# !pip install ultralytics roboflow opencv-python-headless
# !pip install roboflow


from roboflow import Roboflow

try:
    rf = Roboflow(api_key="")
    project = rf.workspace("darnisas-workspace").project("data-trafficgithub")
    version = project.version(1)
    dataset = version.download("yolov8")
    print("Dataset berhasil diunduh!")
except Exception as e:
    print(f"Gagal mendownload dataset. Cek API Key/Nama Project. Error: {e}")
    # Berhenti jika gagal data
    raise 

# ============================================================
# TRAINING MODEL (Dual GPU T4 x2)
# ============================================================
print("\n=== STEP 2: MULAI TRAINING MODEL (DUAL GPU) ===")
from ultralytics import YOLO
import torch

# Pastikan menggunakan GPU
device = '0,1' if torch.cuda.device_count() > 1 else '0'
print(f"Menggunakan Device GPU: {device}")

# YOLOv8s (Small) agar efisien dengan 16k data
model_train = YOLO('yolov8s.pt')

# Training optimasi (Cache, Workers, Dual GPU)
model_train.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,               
    imgsz=416,               # Optimasi kecepatan
    batch=32,                # Auto-batch maksimal GPU
    device=device,           # Gunakan [0,1] untuk dual GPU
    project=OUTPUT_MODEL_DIR,
    name='etle_model',
    cache=True,
    workers=4,
    amp=True,
    patience=5               # Early stopping jika tidak ada peningkatan
)

print(f"Training Selesai. Model disimpan di: {OUTPUT_MODEL_DIR}/etle_model/weights/best.pt")


Skenario 1 - Ada lampu lalu lintas dan zebra cross/line stop

In [ ]:
import os
import glob
import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO

# ==========================================
# 1. SETUP
# ==========================================
WORKING_DIR = "/kaggle/working"
OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, "result.mp4")
trained_model_path = "/kaggle/working/train_output/etle_model-2/weights/best.pt"


video_files = glob.glob('/kaggle/input/**/*.mp4', recursive=True)
INPUT_VIDEO_PATH = video_files[0]

model_standard = YOLO("yolov8m.pt") 
model_custom = YOLO(trained_model_path)

# ==========================================
# 2. LOGIKA WARNA
# ==========================================
def get_traffic_light_state(crop_img):
    if crop_img.size == 0: return "UNKNOWN"
    blurred = cv2.GaussianBlur(crop_img, (5, 5), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    
    mask_red1 = cv2.inRange(hsv, np.array([0, 40, 120]), np.array([10, 255, 255]))
    mask_red2 = cv2.inRange(hsv, np.array([160, 40, 120]), np.array([180, 255, 255]))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)
    mask_green = cv2.inRange(hsv, np.array([40, 40, 120]), np.array([90, 255, 255]))
    
    red_cnt, green_cnt = cv2.countNonZero(mask_red), cv2.countNonZero(mask_green)
    
    if red_cnt > green_cnt and red_cnt > 10: return "RED"
    elif green_cnt > red_cnt and green_cnt > 10: return "GREEN"
    return "UNKNOWN"

# ==========================================
# 3. PROSES VIDEO
# ==========================================
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# Menyimpan riwayat pergerakan pusat kendaraan untuk mengetahui arah
vehicle_trajectory = defaultdict(lambda: deque(maxlen=15)) 
vehicle_zone_status = defaultdict(lambda: False)
violation_ids = set()

zone_light_history = defaultdict(lambda: deque(maxlen=20))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # A. DETEKSI ZEBRA CROSS / STOP LINE (Conf diturunkan agar lebih sensitif)
    results_cw = model_custom.predict(frame, conf=0.35, verbose=False)
    zones = []
    
    for r in results_cw:
        for box in r.boxes:
            zx1, zy1, zx2, zy2 = map(int, box.xyxy[0])
            zc_x, zc_y = int((zx1 + zx2) / 2), int((zy1 + zy2) / 2)
            zones.append({'box': (zx1, zy1, zx2, zy2), 'center': (zc_x, zc_y), 'light_state': "UNKNOWN", 'best_light_pos': None})
            
            cv2.rectangle(frame, (zx1, zy1), (zx2, zy2), (255, 255, 0), 2)
            cv2.putText(frame, "ZONE", (zx1, zy1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # B. DETEKSI LAMPU & KENDARAAN
    results_std = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
    
    if results_std[0].boxes is not None:
        boxes = results_std[0].boxes.xyxy.cpu().numpy()
        class_ids = results_std[0].boxes.cls.cpu().numpy().astype(int)
        
        # 1. Kumpulkan Lampu Lalu Lintas
        lights = []
        for box, cls_id in zip(boxes, class_ids):
            if cls_id == 9: # Traffic Light
                lx1, ly1, lx2, ly2 = map(int, box)
                lc_x, lc_y = int((lx1 + lx2)/2), int((ly1 + ly2)/2)
                state = get_traffic_light_state(frame[ly1:ly2, lx1:lx2])
                lights.append({'center': (lc_x, lc_y), 'state': state})
                cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), (0, 165, 255), 2)

        # 2. ASOSIASI SPASIAL (Mencari lampu yang berada "Di Arah Depan" Zona)
        for idx, zone in enumerate(zones):
            zx, zy = zone['center']
            best_score = float('inf')
            best_light = None
            
            for light in lights:
                lx, ly = light['center']
                # LOGIKA ARAH: Lampu biasanya berada di ATAS zona (ly < zy) pada gambar 2D kamera
                if ly < zy: 
                    # Hitung kecocokan: Semakin sejajar (dx kecil), semakin baik
                    dx = abs(lx - zx)
                    dy = abs(ly - zy)
                    score = dx + (dy * 0.2) # Memberi bobot lebih pada kesejajaran horizontal
                    
                    if score < best_score:
                        best_score = score
                        best_light = light
            
            if best_light:
                zone_light_history[idx].append(best_light['state'])
                zone['light_state'] = max(set(zone_light_history[idx]), key=zone_light_history[idx].count)
                zone['best_light_pos'] = best_light['center']
                
                # GAMBAR GARIS PENGHUBUNG ZONA KE LAMPU
                cv2.line(frame, (zx, zy), best_light['center'], (255, 255, 255), 1)
                
            l_color = (0,0,255) if zone['light_state']=="RED" else (0,255,0) if zone['light_state']=="GREEN" else (0,255,255)
            cv2.putText(frame, f"TL:{zone['light_state']}", (zone['box'][0], zone['box'][1]+20), 0, 0.6, l_color, 2)

        # 3. EVALUASI PERGERAKAN KENDARAAN
        if results_std[0].boxes.id is not None:
            track_ids = results_std[0].boxes.id.cpu().numpy().astype(int)
            
            for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
                if cls_id in [2, 3, 5, 7]:
                    vx1, vy1, vx2, vy2 = map(int, box)
                    v_center_x, v_bottom_y = int((vx1+vx2)/2), vy2
                    
                    # Simpan jejak kendaraan (Trajectory)
                    vehicle_trajectory[track_id].append((v_center_x, v_bottom_y))
                    
                    # Tentukan arah (bergerak ke ATAS/Menjauh atau ke BAWAH/Mendekat)
                    moving_forward = False
                    if len(vehicle_trajectory[track_id]) > 5:
                        start_y = vehicle_trajectory[track_id][0][1]
                        current_y = vehicle_trajectory[track_id][-1][1]
                        # Jika Y mengecil, mobil bergerak menjauh ke arah horizon (Maju)
                        if (start_y - current_y) > 5: 
                            moving_forward = True

                    # Cek pelanggaran di setiap zona
                    for zone in zones:
                        zx1, zy1, zx2, zy2 = zone['box']
                        
                        # Apakah roda mobil menyentuh zona ini?
                        is_inside = (zx1 <= v_center_x <= zx2) and (zy1 <= v_bottom_y <= zy2)
                        was_inside = vehicle_zone_status[track_id]
                        
                        # LOGIKA TILANG: 
                        # Lampu Merah + Masuk Garis + Bergerak Maju (menuju lampu)
                        if zone['light_state'] == "RED" and is_inside and not was_inside and moving_forward:
                            violation_ids.add(track_id)
                        
                        if is_inside:
                            vehicle_zone_status[track_id] = True
                    
                    # Reset status jika mobil sudah jauh dari semua zona (untuk pembersihan memori)
                    is_currently_in_any = any((z['box'][0] <= v_center_x <= z['box'][2]) and (z['box'][1] <= v_bottom_y <= z['box'][3]) for z in zones)
                    if not is_currently_in_any:
                        vehicle_zone_status[track_id] = False

                    # Render Bounding Box Kendaraan
                    is_violator = track_id in violation_ids
                    color = (0, 0, 255) if is_violator else (0, 255, 0)
                    cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), color, 2)
                    
                    # Render tanda panah arah pergerakan mobil
                    if moving_forward:
                        cv2.arrowedLine(frame, (v_center_x, v_bottom_y), (v_center_x, v_bottom_y - 20), (255, 255, 255), 2, tipLength=0.5)

    out.write(frame)

cap.release()
out.release()
print("\n🎉 Proses Selesai! Cek video output.")

Skenario 2 - Kamera Dinamis atau tanpa zebra cross/line stop tapi ada lampu lalu lintas

In [ ]:
import os
import glob
import cv2
import numpy as np
import time
from collections import defaultdict, deque
from ultralytics import YOLO

# ==========================================
# 1. SETUP
# ==========================================
WORKING_DIR = "/kaggle/working"
trained_model_path = "/kaggle/working/train_output/etle_model/weights/best.pt"

video_files = glob.glob('/kaggle/input/**/*.mp4', recursive=True)

if not video_files:
    print("⚠️ Peringatan: Tidak ada file video yang ditemukan!")
else:
    print(f"🎬 Ditemukan {len(video_files)} video untuk diproses.")

model_custom = YOLO(trained_model_path)
model_standard = YOLO("yolov8m.pt") 

evaluation_metrics = {}

# ==========================================
# 2. DETEKSI WARNA LAMPU
# ==========================================
def get_traffic_light_state(crop_img):
    if crop_img.size == 0: return "UNKNOWN"
    blurred = cv2.GaussianBlur(crop_img, (5, 5), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    
    mask_red1 = cv2.inRange(hsv, np.array([0, 40, 120]), np.array([10, 255, 255]))
    mask_red2 = cv2.inRange(hsv, np.array([160, 40, 120]), np.array([180, 255, 255]))
    mask_red = cv2.bitwise_or(mask_red1, mask_red2)
    mask_green = cv2.inRange(hsv, np.array([40, 40, 120]), np.array([90, 255, 255]))
    
    r_cnt, g_cnt = cv2.countNonZero(mask_red), cv2.countNonZero(mask_green)
    if r_cnt > g_cnt and r_cnt > 10: return "RED"
    elif g_cnt > r_cnt and g_cnt > 10: return "GREEN"
    return "UNKNOWN"

# ==========================================
# 3. PROSES BATCH VIDEO
# ==========================================
for INPUT_VIDEO_PATH in video_files:
    video_filename = os.path.basename(INPUT_VIDEO_PATH)
    OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, f"result_{video_filename}")
    
    print(f"\n⏳ Memproses Video: {video_filename}...")
    start_time = time.time()
    
    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps if fps > 0 else 30, (w, h))

    vehicle_trajectory = defaultdict(lambda: deque(maxlen=15)) 
    violation_ids = set()
    light_history = defaultdict(lambda: deque(maxlen=20))
    inferred_state_history = defaultdict(lambda: deque(maxlen=30)) # Stabilizer untuk deduksi lampu
    
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        # A. DETEKSI ZONA (Model: Zebra Cross / Stop Line)
        results_cw = model_custom.predict(frame, conf=0.30, verbose=False)
        detected_zones = []
        if results_cw[0].boxes is not None:
            for box in results_cw[0].boxes:
                zx1, zy1, zx2, zy2 = map(int, box.xyxy[0])
                detected_zones.append({'box': (zx1, zy1, zx2, zy2)})

        # B. DETEKSI KENDARAAN & LAMPU
        results_std = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
        
        lights = []
        vehicles = [] # Simpan data kendaraan untuk diproses deduksi lajur
        
        if results_std[0].boxes is not None:
            boxes_std = results_std[0].boxes.xyxy.cpu().numpy()
            classes_std = results_std[0].boxes.cls.cpu().numpy().astype(int)
            t_ids = results_std[0].boxes.id.cpu().numpy().astype(int) if results_std[0].boxes.id is not None else []
            
            for box, cls, t_id in zip(boxes_std, classes_std, t_ids if len(t_ids) > 0 else [-1]*len(boxes_std)):
                if cls == 9: # Traffic Light
                    lx1, ly1, lx2, ly2 = map(int, box)
                    state = get_traffic_light_state(frame[ly1:ly2, lx1:lx2])
                    lights.append({'box': (lx1, ly1, lx2, ly2), 'center': ((lx1+lx2)//2, ly2), 'state': state})
                elif cls in [2, 3, 5, 7] and t_id != -1: # Kendaraan
                    vx1, vy1, vx2, vy2 = map(int, box)
                    v_cx, v_by = (vx1+vx2)//2, vy2
                    vehicle_trajectory[t_id].append((v_cx, v_by))
                    
                    is_moving_forward = False
                    if len(vehicle_trajectory[t_id]) >= 3:
                        prev_y = vehicle_trajectory[t_id][-3][1]
                        is_moving_forward = (prev_y - v_by) > 2  # Kecepatan bergerak
                        
                    vehicles.append({'id': t_id, 'box': (vx1, vy1, vx2, vy2), 'center_bottom': (v_cx, v_by), 'moving': is_moving_forward})

        # C. PEMASANGAN LAMPU & DEDUKSI KONDISI LAJUR (INFERENCE MODE)
        active_violation_zones = []
        claimed_zone_indices = set()

        # Tahap 1: Pemasangan dengan lampu fisik (Jika ada)
        for light in lights:
            lc_x, lc_y = light['center']
            s_id = f"L_{lc_x//50}_{lc_y//50}"
            light_history[s_id].append(light['state'])
            stable_state = max(set(light_history[s_id]), key=light_history[s_id].count)

            matched_zone_idx = -1
            v_box = None
            
            for idx, zone in enumerate(detected_zones):
                zx1, zy1, zx2, zy2 = zone['box']
                zc_x = (zx1 + zx2) // 2
                
                # ISOLASI LAJUR KETAT
                if abs(zc_x - lc_x) < (w * 0.20):
                    if lc_y < zy1 and (zy1 - lc_y) < (h * 0.45):
                        v_box = zone['box']
                        matched_zone_idx = idx
                        break
            
            if v_box is not None:
                claimed_zone_indices.add(matched_zone_idx)
                cv2.rectangle(frame, (v_box[0], v_box[1]), (v_box[2], v_box[3]), (255, 255, 0), 2)
                cv2.putText(frame, "REAL ZONE", (v_box[0], v_box[1] - 5), 0, 0.5, (255, 255, 0), 2)
            else:
                # VIRTUAL BOUNDING BOX
                v_width = int(w * 0.25)
                v_y1 = lc_y + int(h * 0.25)
                v_y2 = v_y1 + int(h * 0.08)
                v_y1, v_y2 = min(v_y1, h - 5), min(v_y2, h)
                v_x1 = lc_x - (v_width // 2)
                v_x2 = lc_x + (v_width // 2)
                v_box = (v_x1, v_y1, v_x2, v_y2)
                cv2.rectangle(frame, (v_x1, v_y1), (v_x2, v_y2), (255, 100, 0), 2)
                cv2.putText(frame, "VIRTUAL ZONE", (v_x1, v_y1 - 5), 0, 0.4, (255, 100, 0), 2)

            active_violation_zones.append({'box': v_box, 'state': stable_state, 'source': 'PHYSICAL'})
            
            # Gambar Kotak Lampu Fisik
            l_col = (0,0,255) if stable_state == "RED" else (0,255,0)
            cv2.rectangle(frame, (light['box'][0], light['box'][1]), (light['box'][2], light['box'][3]), l_col, 2)
            cv2.putText(frame, stable_state, (light['box'][0], light['box'][1]-10), 0, 0.6, l_col, 2)

        # Tahap 2: Inference Mode untuk Zebra Cross tanpa lampu (Top-down view / Silau)
        for idx, zone in enumerate(detected_zones):
            if idx not in claimed_zone_indices:
                zx1, zy1, zx2, zy2 = zone['box']
                
                # Tentukan Area Antrean (Area tepat di bawah / sebelum garis stop)
                # Estimasi sejauh 25% tinggi layar ke bawah dari garis Zebra Cross
                queue_area_y_end = min(zy1 + int(h * 0.25), h)
                
                stop_count = 0
                move_count = 0
                
                for v in vehicles:
                    v_cx, v_by = v['center_bottom']
                    # Cek apakah kendaraan berada di dalam jalur dan di area antrean
                    if zx1 <= v_cx <= zx2 and zy1 < v_by < queue_area_y_end:
                        if v['moving']:
                            move_count += 1
                        else:
                            stop_count += 1
                
                # Logika Mayoritas: Jika yang berhenti lebih banyak dari yang jalan = MERAH
                current_inferred_state = "GREEN"
                if stop_count > move_count and stop_count >= 1: 
                    current_inferred_state = "RED"
                
                # Stabilisasi Keputusan Inferensi agar tidak berkedip tiap frame
                z_id = f"Z_INF_{zx1}_{zy1}"
                inferred_state_history[z_id].append(current_inferred_state)
                stable_inferred = max(set(inferred_state_history[z_id]), key=inferred_state_history[z_id].count)

                active_violation_zones.append({'box': zone['box'], 'state': stable_inferred, 'source': 'INFERRED'})
                
                # Gambar Zona Hasil Deduksi
                z_col = (0,0,255) if stable_inferred == "RED" else (0,255,0)
                cv2.rectangle(frame, (zx1, zy1), (zx2, zy2), z_col, 2)
                cv2.putText(frame, f"INFERRED: {stable_inferred}", (zx1, zy1 - 5), 0, 0.5, z_col, 2)

        # D. EVALUASI PELANGGARAN KENDARAAN
        for v in vehicles:
            t_id = v['id']
            vx1, vy1, vx2, vy2 = v['box']
            v_cx, v_by = v['center_bottom']
            is_moving_forward = v['moving']
            
            if len(vehicle_trajectory[t_id]) >= 3:
                prev_y = vehicle_trajectory[t_id][-3][1]
                
                for zone_info in active_violation_zones:
                    zx1, zy1, zx2, zy2 = zone_info['box']
                    
                    if zx1 <= v_cx <= zx2: # Di dalam lajur
                        if zone_info['state'] == "RED":
                            
                            crossed_line = (prev_y > zy1 and v_by <= zy1)
                            moving_inside_zone = (zy1 <= v_by <= zy2 and is_moving_forward)
                            moving_past_zone = (v_by < zy1 and prev_y <= zy1 and is_moving_forward) 
                            
                            if crossed_line or moving_inside_zone or moving_past_zone:
                                violation_ids.add(t_id)

            # Tampilkan Status Kendaraan
            color = (0, 0, 255) if t_id in violation_ids else (0, 255, 0)
            cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), color, 2)
            if t_id in violation_ids:
                cv2.putText(frame, "PELANGGARAN!", (vx1, vy1-10), 0, 0.6, (0,0,255), 2)

        cv2.putText(frame, f"Pelanggaran: {len(violation_ids)}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        out.write(frame)

    cap.release()
    out.release()
    
    end_time = time.time()
    processing_time = end_time - start_time
    avg_fps = frame_count / processing_time if processing_time > 0 else 0
    
    evaluation_metrics[video_filename] = {
        'total_pelanggaran': len(violation_ids),
        'total_frame': frame_count,
        'waktu_proses_detik': round(processing_time, 2),
        'fps_rata_rata': round(avg_fps, 2)
    }
    print(f"✅ Selesai: {video_filename} | Pelanggaran Tepat: {len(violation_ids)} | Kecepatan: {round(avg_fps, 2)} FPS")

# ==========================================
# 4. CETAK LAPORAN METRIK EVALUASI
# ==========================================
print("\n" + "="*60)
print("📊 LAPORAN METRIK EVALUASI ETLE")
print("="*60)
print(f"{'Nama Video':<30} | {'Pelanggaran':<11} | {'Waktu (s)':<9} | {'FPS':<6}")
print("-" * 60)

total_semua_pelanggaran = 0
for vid, metrics in evaluation_metrics.items():
    print(f"{vid:<30} | {metrics['total_pelanggaran']:<11} | {metrics['waktu_proses_detik']:<9} | {metrics['fps_rata_rata']:<6}")
    total_semua_pelanggaran += metrics['total_pelanggaran']

print("-" * 60)
print(f"TOTAL PELANGGARAN DI SEMUA VIDEO: {total_semua_pelanggaran}")
print("="*60)
print("🎉 Proses Batch dengan Logical Inference Selesai!")

Skenario 3 - Pure Behavior

In [ ]:
import os
import glob
import cv2
import math
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO

print("\n=====================================================================")
print("🚀 PURE BEHAVIORAL
print("=====================================================================")

# ==========================================
# 1. SETUP
# ==========================================
WORKING_DIR = "/kaggle/working"
video_files = glob.glob('/kaggle/input/**/*.mp4', recursive=True)
if not video_files:
    print("⚠️ File video .mp4 tidak ditemukan!")
    exit()

INPUT_VIDEO_PATH = video_files[0]
video_filename = os.path.basename(INPUT_VIDEO_PATH)
OUTPUT_VIDEO_PATH = os.path.join(WORKING_DIR, f"pure_behavior_{video_filename}")

# --- FOLDER UNTUK MENYIMPAN GAMBAR PELANGGARAN ---
VIOLATION_DIR = os.path.join(WORKING_DIR, "violations")
os.makedirs(VIOLATION_DIR, exist_ok=True)
print(f"📁 Folder penyimpanan capture pelanggaran disiapkan di: {VIOLATION_DIR}")

# Gunakan model YOLO standar
model_standard = YOLO("yolov8m.pt") 

# ==========================================
# 2. MEMORY ACCUMULATOR
# ==========================================
vehicle_registry = defaultdict(lambda: deque(maxlen=30))
smoothed_positions = {}
stopped_frame_counter = defaultdict(int)
confirmed_violations = set()

# Menyimpan posisi garis terakhir yang valid per lajur (direction) agar tidak collapse saat konvoi melanggar
persistent_anchors = {} 
ANCHOR_TTL_FRAMES = 60  # Batas umur garis virtual (2 detik jika video ~30fps) setelah antrean cair

# PARAMETER LOGIKA BEHAVIOR
PROXIMITY_THRESHOLD = 0.35  # Radius lingkungan terdekat (35% dari lebar frame)
STOP_SPEED_THRESHOLD = 0.5  # Kecepatan di bawah ini dianggap diam/berhenti
STOP_DURATION_FRAMES = 10   # Harus diam 10 frame berturut-turut agar sah jadi Anchor
MOVING_SPEED_THRESHOLD = 1.5 # Kecepatan melaju (menerobos)
LINE_EXTENT = 250           # Seberapa panjang garis batas ditarik ke kiri & kanan (piksel)

# PARAMETER PEMBATAS SEGMEN (NON-KENDARAAN)
SEGMENT_WIDTH_THRESHOLD = 0.15 # Maksimal lebar lajur (15% dari lebar frame)

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps if fps > 0 else 30, (w, h))

frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    
    # Track semua kendaraan (Mobil, Motor, Bus, Truk)
    results = model_standard.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)
    
    active_vehicles = []
    
    if results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else []
        
        for idx, (box, cls) in enumerate(zip(boxes, classes)):
            # 2: car, 3: motorcycle, 5: bus, 7: truck
            if cls in [2, 3, 5, 7] and idx < len(track_ids): 
                t_id = track_ids[idx]
                x1, y1, x2, y2 = map(int, box)
                
                # Titik tumpu ban bawah kendaraan
                raw_cx, raw_cy = (x1 + x2) // 2, y2 
                
                # EMA Smoothing untuk posisi
                if t_id not in smoothed_positions: 
                    smoothed_positions[t_id] = (raw_cx, raw_cy)
                else:
                    scx, scy = smoothed_positions[t_id]
                    smoothed_positions[t_id] = (int(scx * 0.7 + raw_cx * 0.3), int(scy * 0.7 + raw_cy * 0.3))
                
                cx, cy = smoothed_positions[t_id]
                v_hist = vehicle_registry[t_id]
                
                # Kinematika Arah
                speed, dominant_direction = 0.0, "UNKNOWN"
                if len(v_hist) > 0:
                    past_cx, past_cy = v_hist[0]['cx'], v_hist[0]['cy']
                    dy, dx = cy - past_cy, cx - past_cx
                    speed = math.sqrt(dx**2 + dy**2) / len(v_hist)
                    
                    if abs(dy) > abs(dx):
                        dominant_direction = "UP_STREAM" if dy < 0 else "DOWN_STREAM"
                    else:
                        dominant_direction = "LEFT_STREAM" if dx < 0 else "RIGHT_STREAM"
                
                # Syarat Durasi Berhenti
                if speed < STOP_SPEED_THRESHOLD:
                    stopped_frame_counter[t_id] += 1
                else:
                    stopped_frame_counter[t_id] = max(0, stopped_frame_counter[t_id] - 2)
                    
                # [PERUBAHAN MASALAH 1]: Jika kendaraan sudah mulai jalan (speed >= threshold), 
                # status is_fully_stopped langsung Gugur agar konsensus cepat merespon lampu hijau.
                is_fully_stopped = (stopped_frame_counter[t_id] >= STOP_DURATION_FRAMES) and (speed < STOP_SPEED_THRESHOLD)
                
                v_hist.append({'cx': cx, 'cy': cy, 'speed': speed, 'dir': dominant_direction})
                
                active_vehicles.append({
                    'id': t_id, 'box': (x1, y1, x2, y2), 'center': (cx, cy),
                    'speed': speed, 'direction': dominant_direction, 'is_stopped': is_fully_stopped
                })

    # ==========================================
    # 3. DYNAMIC NEIGHBORHOOD ANCHOR LOGIC
    # ==========================================
    processed_ids = set()
    dynamic_anchors = [] 

    for v in active_vehicles:
        if v['id'] in processed_ids: continue
        
        neighborhood = []
        for peer in active_vehicles:
            if peer['direction'] == v['direction'] and v['direction'] != "UNKNOWN":
                
                dx = v['center'][0] - peer['center'][0]
                dy = v['center'][1] - peer['center'][1]
                dist = math.sqrt(dx**2 + dy**2)
                
                if dist < (w * PROXIMITY_THRESHOLD):
                    
                    is_in_same_segment = True
                    
                    if v['direction'] in ["UP_STREAM", "DOWN_STREAM"]:
                        if abs(dx) > (w * SEGMENT_WIDTH_THRESHOLD):
                            is_in_same_segment = False
                    else:
                        if abs(dy) > (h * SEGMENT_WIDTH_THRESHOLD):
                            is_in_same_segment = False
                    
                    if is_in_same_segment:
                        neighborhood.append(peer)
                        processed_ids.add(peer['id'])
                        
        # --- LOGIKA KONSENSUS ANCHOR ---
        stopped_cars = [c for c in neighborhood if c['is_stopped']]
        moving_cars = [c for c in neighborhood if not c['is_stopped']]
        
        if len(stopped_cars) >= 2 and len(stopped_cars) >= len(moving_cars):
            direction = v['direction']
            leader_car = None
            
            if direction == "UP_STREAM": 
                leader_car = min(stopped_cars, key=lambda c: c['center'][1])
            elif direction == "DOWN_STREAM": 
                leader_car = max(stopped_cars, key=lambda c: c['center'][1])
            elif direction == "LEFT_STREAM": 
                leader_car = min(stopped_cars, key=lambda c: c['center'][0])
            elif direction == "RIGHT_STREAM": 
                leader_car = max(stopped_cars, key=lambda c: c['center'][0])
                
            if leader_car:
                anchor_pos = leader_car['center']
                dynamic_anchors.append({'direction': direction, 'pos': anchor_pos, 'neighborhood': neighborhood})
                
                # Kunci koordinat ke dalam Persistent Registry & Refresh TTL ---
                persistent_anchors[direction] = {
                    'pos': anchor_pos,
                    'ttl': ANCHOR_TTL_FRAMES
                }

    # MANAGEMENT LOGIKA SIKLUS HIDUP PERSISTENT ANCHOR ---
    # Jika dynamic_anchors di frame ini tidak mendeteksi garis (antrean runtuh), gunakan data dari persistent_anchors
    active_directions_this_frame = [a['direction'] for a in dynamic_anchors]
    
    for direction, p_data in list(persistent_anchors.items()):
        # Kurangi waktu hitung mundur jika di frame ini konsensus visual lajur tersebut kosong/runtuh
        if direction not in active_directions_this_frame:
            p_data['ttl'] -= 1
            if p_data['ttl'] <= 0:
                del persistent_anchors[direction] # Hapus permanen jika waktu habis (Konfirmasi Lampu Hijau)
                continue
        
        # Gambar Garis dari memori cadangan jika visual aslinya sempat hilang
        if direction not in active_directions_this_frame:
            ax, ay = p_data['pos']
            cv2.line(frame, (ax - LINE_EXTENT, ay), (ax + LINE_EXTENT, ay), (0, 64, 255), 2) # Orange kemerahan untuk garis memori
            cv2.putText(frame, f"STOP LINE MEMORY ({direction})", (ax - LINE_EXTENT + 10, ay - 10), 0, 0.5, (0, 64, 255), 1)

    # ==========================================
    # 4. VIOLATION CHECK AGAINST ACTIVE ANCHORS
    # ==========================================
    for v in active_vehicles:
        t_id = v['id']
        x1, y1, x2, y2 = v['box']
        cx, cy = v['center']
        
        if t_id in confirmed_violations:
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(frame, f"VIOLATOR #{t_id}", (x1, y1 - 10), 0, 0.5, (0, 0, 255), 2)
            continue
            
        # MERGE CHECK MENGGUNAKAN PERSISTENT MEMORY ---
        # Mengambil acuan dari persistent_anchors agar penilangan konvoi kendaraan mengekor tidak bocor
        if v['direction'] in persistent_anchors:
            p_anchor = persistent_anchors[v['direction']]
            ax, ay = p_anchor['pos']
            is_in_same_segment = False
            
            if v['direction'] in ["UP_STREAM", "DOWN_STREAM"]:
                if abs(cx - ax) <= (w * SEGMENT_WIDTH_THRESHOLD):
                    is_in_same_segment = True
            else:
                if abs(cy - ay) <= (h * SEGMENT_WIDTH_THRESHOLD):
                    is_in_same_segment = True
            
            if is_in_same_segment:
                if not v['is_stopped'] and v['speed'] > MOVING_SPEED_THRESHOLD:
                    
                    # [PERUBAHAN MASALAH 1]: Proteksi Lampu Hijau. jika stopped_frame_counter > 0, 
                    # artinya kendaraan ini baru saja melepas rem dari antrean (bukan nerobos dari belakang saat merah).
                    if stopped_frame_counter[t_id] > 0:
                        continue
                        
                    past_cy = vehicle_registry[t_id][0]['cy'] if len(vehicle_registry[t_id]) >= 5 else cy
                    past_cx = vehicle_registry[t_id][0]['cx'] if len(vehicle_registry[t_id]) >= 5 else cx
                    
                    is_crossed = False
                    
                    if v['direction'] == "UP_STREAM" and past_cy >= ay and cy < ay: 
                        is_crossed = True
                    elif v['direction'] == "DOWN_STREAM" and past_cy <= ay and cy > ay: 
                        is_crossed = True
                    elif v['direction'] == "LEFT_STREAM" and past_cx >= ax and cx < ax: 
                        is_crossed = True
                    elif v['direction'] == "RIGHT_STREAM" and past_cx <= ax and cx > ax: 
                        is_crossed = True
                    
                    if is_crossed:
                        confirmed_violations.add(t_id)
                        
                        # Gambar kotak merah sebelum dicapture agar terlihat jelas di foto
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        cv2.putText(frame, f"VIOLATOR #{t_id}", (x1, y1 - 10), 0, 0.5, (0, 0, 255), 2)
                        
                        # Simpan frame ke dalam folder violations
                        capture_filename = os.path.join(VIOLATION_DIR, f"violation_id{t_id}_frame{frame_idx}.jpg")
                        cv2.imwrite(capture_filename, frame)
                        print(f"📸 Captured violation for ID {t_id} at frame {frame_idx}")

        # Render kotak normal
        if t_id not in confirmed_violations:
            color = (0, 255, 0) if not v['is_stopped'] else (255, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # ==========================================
    # HUD DASHBOARD
    # ==========================================
    cv2.rectangle(frame, (15, 15), (510, 85), (0, 0, 0), -1)
    cv2.rectangle(frame, (15, 15), (510, 85), (255, 128, 0), 2)
    cv2.putText(frame, "ETLE V7.3: PERSISTENT BEHAVIORAL CONSENSUS", (25, 40), 0, 0.5, (255, 128, 0), 2)
    cv2.putText(frame, f"ACTIVE ANCHORS: {len(persistent_anchors)} | VIOLATIONS: {len(confirmed_violations)}", (25, 70), 0, 0.5, (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()
print(f"\n✅ PROSES SELESAI!")
print(f"🎥 Video tersimpan di: {OUTPUT_VIDEO_PATH}")
print(f"📸 Semua gambar hasil capture ada di folder: {VIOLATION_DIR}")